# 🐾 Cats vs Dogs Image Classification — Deep Transfer Learning (MobileNetV2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rshamith777-cpu/AIML_1_ImageClassification_byte/blob/main/notebooks/cats_vs_dogs_transfer_learning.ipynb)

**Repository:** `AIML_1_ImageClassification_byte`  
**Track:** Arithmatrix Virtual Internship Program (AVIP 2026) — AI/ML Track (Task 1)  
**Architecture:** MobileNetV2 (ImageNet Pre-trained) + GlobalAveragePooling2D + Fine-Tuning Head  
**Dataset:** Real Microsoft / Kaggle Dogs vs. Cats Benchmark (25,000 RGB Photographs) via `tensorflow_datasets`  

---

### Step 1: Environment Setup & GPU Verification
Verify GPU availability (e.g., NVIDIA Tesla T4 on Google Colab) and import deep learning libraries.

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPUs Available:", len(gpus))
if gpus:
    print("Device:", tf.test.gpu_device_name())
else:
    print("Note: To enable GPU in Colab, go to Runtime > Change runtime type > T4 GPU.")

### Step 2: Load Real 25,000-Image Dataset via TFDS
We load the official Microsoft/Kaggle `cats_vs_dogs` dataset and split into:
- **80% Training** (20,000 images)
- **10% Validation** (2,500 images)
- **10% Test** (2,500 images)

In [ ]:
(raw_train, raw_val, raw_test), metadata = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    with_info=True,
    as_supervised=True,
)

print("Dataset Metadata:")
print(f"  Training instances  : {len(raw_train)}")
print(f"  Validation instances: {len(raw_val)}")
print(f"  Testing instances   : {len(raw_test)}")
print(f"  Class labels        : {metadata.features['label'].names} (0 = Cat, 1 = Dog)")

### Step 3: Data Preprocessing & Augmentation Pipeline
Images are resized to 160x160 and normalized to [-1, 1] using `mobilenet_v2.preprocess_input`.
Data augmentation (random horizontal flips and rotations) is applied to the training set to prevent overfitting.

In [ ]:
IMG_SIZE = (160, 160)
BATCH_SIZE = 32

def preprocess_image(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
])

train_ds = raw_train.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.cache().shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = raw_val.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).cache().prefetch(tf.data.AUTOTUNE)

test_ds = raw_test.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).cache().prefetch(tf.data.AUTOTUNE)
print("Data pipelines cached and batched successfully.")

### Step 4: Transfer Learning Architecture (MobileNetV2)
We load MobileNetV2 with pre-trained ImageNet weights with its top classification layer removed.
Phase 1: Freeze the base convolutional layers to preserve pre-trained feature detectors.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(160, 160, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1)(x)  # Linear logit output (positive = Dog, negative = Cat)

model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=['accuracy']
)
model.summary()

### Step 5: Phase 1 Training (Feature Extraction)
Train top classification head for 3 epochs.

In [ ]:
history_phase1 = model.fit(
    train_ds,
    epochs=3,
    validation_data=val_ds
)

### Step 6: Phase 2 Fine-Tuning
Unfreeze the top 30 convolutional layers of MobileNetV2 and train with a lower learning rate ($10^{-5}$) for 2 additional epochs.

In [ ]:
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=['accuracy']
)

history_phase2 = model.fit(
    train_ds,
    epochs=5,
    initial_epoch=history_phase1.epoch[-1],
    validation_data=val_ds
)

### Step 7: Final Test Set Evaluation & Metrics Export
Evaluate on the 2,500-sample test set. Compute Accuracy, Precision, Recall, F1, and save `confusion_matrix.png` and `metrics.json`.

In [ ]:
y_true, y_pred_probs = [], []
for batch_x, batch_y in test_ds:
    logits = model.predict(batch_x, verbose=0)
    probs = tf.sigmoid(logits).numpy().flatten()
    y_pred_probs.extend(probs)
    y_true.extend(batch_y.numpy())

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
y_pred = (y_pred_probs >= 0.50).astype(int)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print("=== FINAL TEST RESULTS ===")
print(f"Accuracy : {acc*100:.2f}%")
print(f"Precision: {prec*100:.2f}%")
print(f"Recall   : {rec*100:.2f}%")
print(f"F1-Score : {f1*100:.2f}%")
print("\nConfusion Matrix:\n", cm)

# Save metrics.json
os.makedirs('outputs', exist_ok=True)
metrics_dict = {
    'accuracy': round(float(acc), 4),
    'precision_dog': round(float(prec), 4),
    'recall_dog': round(float(rec), 4),
    'f1_dog': round(float(f1), 4),
    'test_samples': len(y_true)
}
with open('outputs/metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=2)

# Plot Confusion Matrix
import seaborn as sns
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Cat', 'Dog'], yticklabels=['Cat', 'Dog'])
plt.xlabel('Predicted Label')
plt.ylabel('Ground Truth')
plt.title('Confusion Matrix — Cats vs Dogs (Test Set)')
plt.savefig('outputs/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

### Step 8: Save 10 Sample Inference Photographs & CSV Table
Extract 10 real test photos (5 cats, 5 dogs) from `test_ds` and export them with predictions.

In [ ]:
import csv
from PIL import Image

os.makedirs('outputs/sample_inferences', exist_ok=True)
samples_data = []
cat_count, dog_count = 0, 0

for img_raw, label_raw in raw_test:
    lbl = int(label_raw.numpy())
    if lbl == 0 and cat_count < 5:
        cat_count += 1
        fname = f'cat_0{cat_count}.jpg'
        target_label = 'cat'
    elif lbl == 1 and dog_count < 5:
        dog_count += 1
        fname = f'dog_0{dog_count}.jpg'
        target_label = 'dog'
    else:
        if cat_count >= 5 and dog_count >= 5:
            break
        continue

    # Save real photo
    fpath = os.path.join('outputs/sample_inferences', fname)
    pil_img = Image.fromarray(img_raw.numpy())
    pil_img.save(fpath, quality=90)

    # Predict with model
    inp = tf.image.resize(img_raw, IMG_SIZE)
    inp = tf.keras.applications.mobilenet_v2.preprocess_input(inp)
    logit = model.predict(tf.expand_dims(inp, 0), verbose=0)[0][0]
    p_dog = float(tf.sigmoid(logit).numpy())
    pred = 'dog' if p_dog >= 0.50 else 'cat'
    conf = p_dog if pred == 'dog' else (1.0 - p_dog)

    samples_data.append([fname, target_label, pred, f"{p_dog:.4f}", f"{conf*100:.2f}%"])

with open('outputs/sample_predictions.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['image_filename', 'ground_truth', 'predicted_label', 'p_dog', 'confidence'])
    writer.writerows(samples_data)

print("Sample Predictions Table:")
for row in samples_data:
    print(row)

### Step 9: Save Serialized Model & Download Artifacts
Save the trained deep learning model as `cats_dogs_mobilenetv2.keras` and download for repo deployment.

In [ ]:
os.makedirs('models', exist_ok=True)
model_path = 'models/cats_dogs_mobilenetv2.keras'
model.save(model_path)
print(f"Saved production model to {model_path} ({os.path.getsize(model_path) / (1024*1024):.2f} MB)")

# If running inside Colab, uncomment below to auto-download to your PC:
# from google.colab import files
# files.download(model_path)
# files.download('outputs/metrics.json')
# files.download('outputs/confusion_matrix.png')
# files.download('outputs/sample_predictions.csv')